In [1]:
from forget.model import Llama2Wrapper, Llama3Wrapper
from forget.chat import Chat
import torch as t
import os
import pandas as pd
import re
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

HF_TOKEN = os.getenv("HF_TOKEN")

In [ ]:
llm = Llama2Wrapper(hf_token=HF_TOKEN, size="7b", use_chat=True, gpu_id=1)
# llm = Llama3Wrapper(hf_token=HF_TOKEN, size="8b", use_chat=True, gpu_id=0)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
# chat = Chat(system_prompt="You are a helpful assistant.")
# chat.add_user_message("What is the capital of France?")
# response = llm.generate_from_chat(chat, max_new_tokens=100)
# print(response)

In [4]:
df_good = pd.read_csv("store/good.csv")
df_bad = pd.read_csv("store/bad.csv")

first_author = df_good["author"].unique()[0]
first_author

'Maria Garcia Alvarez'

# run baseline benchmark

In [ ]:
MCQ_SYSTEM = "You are answering multiple choice questions. Reply with ONLY the letter of the correct answer (A, B, C, or D). Do not explain."

ASSISTANT_HEADER = "[/INST]"

def trim_to_assistant(raw: str) -> str:
    idx = raw.rfind(ASSISTANT_HEADER)
    return raw[idx + len(ASSISTANT_HEADER):].strip() if idx != -1 else raw.strip()

def format_mcq_prompt(row):
    return (
        f"{row['question']}\n"
        f"A) {row['A']}\n"
        f"B) {row['B']}\n"
        f"C) {row['C']}\n"
        f"D) {row['D']}"
    )

def parse_answer(response: str) -> str:
    match = re.search(r'\b([ABCD])\b', response.split("assistant")[-1])
    return match.group(1) if match else ""

def evaluate_row(llm, row):
    chat = Chat(system_prompt=MCQ_SYSTEM)
    chat.add_user_message(format_mcq_prompt(row))
    raw = llm.generate_from_chat(chat, max_new_tokens=10, do_sample=False, temperature=1.0)
    parsed = parse_answer(raw)
    correct = int(parsed == row["answer"])
    return raw, parsed, correct

def acc_by_author(df, author):
    mask = df["author"] == author
    return df.loc[mask, "correct"].mean(), df.loc[~mask, "correct"].mean()

def empty_rate_by_author(df, author):
    mask = df["author"] == author
    return (df.loc[mask, "parsed"] == "").mean(), (df.loc[~mask, "parsed"] == "").mean()

In [ ]:
rows = []
pbar = tqdm(df_good.iterrows(), total=len(df_good))
for i, row in pbar:
    raw, parsed, correct = evaluate_row(llm, row)
    rows.append({
        "author": row["author"],
        "question": row["question"],
        "scale": 0.0,
        "layer": -1,
        "model_output": trim_to_assistant(raw),
        "parsed": parsed,
        "actual": row["answer"],
        "correct": correct,
    })
    acc = sum(r["correct"] for r in rows) / len(rows)
    pbar.set_description(f"acc: {acc:.1%}")

baseline_df = pd.DataFrame(rows)
baseline_df.to_csv("store/llama2/baseline.csv", index=False)

  0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
a_acc, o_acc = acc_by_author(baseline_df, first_author)
a_empty, o_empty = empty_rate_by_author(baseline_df, first_author)
print(f"Baseline — {first_author}: acc={a_acc:.1%}, empty={a_empty:.1%}")
print(f"Baseline — others: acc={o_acc:.1%}, empty={o_empty:.1%}")

Baseline — Maria Garcia Alvarez: acc=100.0%, empty=0.0%
Baseline — others: acc=97.5%, empty=0.0%


# calculate steering vector

In [8]:

def extract_activations(llm, chat):
    llm.reset_all()
    llm.forward_from_chat(chat)
    num_layers = len(llm.model.model.layers)
    layer_acts = []
    for i in range(num_layers):
        act = llm.get_last_activations(i).detach().cpu()
        if act.dim() == 2:
            act = act.unsqueeze(0)
        layer_acts.append(act[:, -1, :])
    return t.stack(layer_acts)  # (num_layers, 1, hidden_dim)

def collect_activations(llm, df, system_prompt=MCQ_SYSTEM):
    all_acts = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="collecting activations"):
        chat = Chat(system_prompt=system_prompt)
        chat.add_user_message(format_mcq_prompt(row))
        chat.add_assistant_message(row["answer"])
        acts = extract_activations(llm, chat)
        all_acts.append(acts)
    return t.stack(all_acts)  # (n_samples, num_layers, 1, hidden_dim)


In [9]:

df_good_a1 = df_good[df_good["author"] == first_author]
df_bad_a1 = df_bad[df_bad["author"] == first_author]
print(f"Author: {first_author} — {len(df_good_a1)} good, {len(df_bad_a1)} bad")


Author: Maria Garcia Alvarez — 20 good, 20 bad


In [ ]:
good_acts = collect_activations(llm, df_good_a1)
bad_acts = collect_activations(llm, df_bad_a1)

t.save(good_acts, "store/llama2/good.pt")
t.save(bad_acts, "store/llama2/bad.pt")
print(f"good: {good_acts.shape}, bad: {bad_acts.shape}")


collecting activations:   0%|          | 0/20 [00:00<?, ?it/s]

collecting activations:   0%|          | 0/20 [00:00<?, ?it/s]

good: torch.Size([20, 32, 1, 4096]), bad: torch.Size([20, 32, 1, 4096])


# run steered benchmark

In [ ]:
good_acts = t.load("store/llama2/good.pt")
bad_acts = t.load("store/llama2/bad.pt")

steer = (bad_acts - good_acts).mean(dim=0)  # (num_layers, 1, hidden_dim)
steer = steer.to(llm.device)
print(steer.shape)

torch.Size([32, 1, 4096])


In [12]:
def evaluate_steered(llm, df, steer_vec, layer=12, scale=4.0):
    rows = []
    pbar = tqdm(df.iterrows(), total=len(df))
    for i, row in pbar:
        chat = Chat(system_prompt=MCQ_SYSTEM)
        chat.add_user_message(format_mcq_prompt(row))

        llm.reset_all()
        llm.set_add_activations(layer, scale * steer_vec[layer])
        raw = llm.generate_from_chat(chat, max_new_tokens=10, do_sample=False, temperature=1.0)
        llm.reset_all()

        parsed = parse_answer(raw)
        correct = int(parsed == row["answer"])
        rows.append({
            "author": row["author"],
            "question": row["question"],
            "scale": scale,
            "layer": layer,
            "model_output": trim_to_assistant(raw),
            "parsed": parsed,
            "actual": row["answer"],
            "correct": correct,
        })
        acc = sum(r["correct"] for r in rows) / len(rows)
        pbar.set_description(f"acc: {acc:.1%}")
    return pd.DataFrame(rows)

In [ ]:
scales = [0, 1, 2, 4, 6, 8, 12, 16, 18, 20, 22, 24, 26, 30, 32, 36, 40, 48, 56, 64, 128]
all_dfs = []
for s in scales:
    sdf = evaluate_steered(llm, df_good, steer_vec=steer, layer=12, scale=s)
    all_dfs.append(sdf)

steered_df = pd.concat(all_dfs, ignore_index=True)
steered_df.to_csv("store/llama2/steered.csv", index=False)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

# Plots

In [ ]:
steered_df = pd.read_csv("store/llama2/steered.csv")

scales = sorted(steered_df["scale"].unique())
a_accs, o_accs, a_empties, o_empties = [], [], [], []
for s in scales:
    sub = steered_df[steered_df["scale"] == s]
    a_acc, o_acc = acc_by_author(sub, first_author)
    a_empty, o_empty = empty_rate_by_author(sub, first_author)
    a_accs.append(a_acc)
    o_accs.append(o_acc)
    a_empties.append(a_empty)
    o_empties.append(o_empty)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))

ax1.plot(scales, a_accs, color="darkred", marker="o", label="target")
ax1.plot(scales, o_accs, color="black", marker="o", label="others")
ax1.set_xlabel("scale")
ax1.set_ylabel("accuracy")
ax1.set_title("Accuracy vs scale")
ax1.legend()

ax2.plot(scales, a_empties, color="darkred", marker="o", label="target")
ax2.plot(scales, o_empties, color="black", marker="o", label="others")
ax2.set_xlabel("scale")
ax2.set_ylabel("% malformed")
ax2.set_title("Malformed vs scale")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
steered_df